## Name: Aryan Mahesh Patil
### Rollno: 46
#### Subject: DL
#### Experiment No: 8

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
texts=[
    "good movie",
    "excellent film",
    "very nice movie",
    "I loved it",
    "amazing film",
    "great movie",
    "fantastic performance",
    "truly enjoyable",
    "captivating story",
    "highly recommend",
    "a really heartwarming tale",
    "the best film I've seen all year",
    "absolutely brilliant acting",
    "a masterpiece of cinema",
    "so glad I watched this",
    "bad movie",
    "terrible film",
    "very boring movie",
    "I hated it",
    "poor film",
    "worst movie",
    "disappointing experience",
    "not worth watching",
    "absolute garbage",
    "couldn't finish it",
    "a complete waste of time",
    "left me feeling cold",
    "the plot was nonsensical",
    "wish I could get my money back",
    "just awful"
]

#positive-1, negative-0
labels=[
    1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,
    0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
]

#=======================================
#Build Vocabulary
#========================================
vocab={"<PAD>":0,"<UNK>":1}

for text in texts:
    for word in text.split():
        if word not in vocab:
            vocab[word]=len(vocab)

def encode(text):
  return [vocab.get(word,1)
          for word in text.lower().split()]

X=[encode(text) for text in texts]

print(X)

[[2, 3], [4, 5], [6, 7, 3], [1, 9, 10], [11, 5], [12, 3], [13, 14], [15, 16], [17, 18], [19, 20], [21, 22, 23, 24], [25, 26, 5, 1, 28, 29, 30], [31, 32, 33], [21, 34, 35, 36], [37, 38, 1, 39, 40], [41, 3], [42, 5], [6, 43, 3], [1, 44, 10], [45, 5], [46, 3], [47, 48], [49, 50, 51], [52, 53], [54, 55, 10], [21, 56, 57, 35, 58], [59, 60, 61, 62], [25, 63, 64, 65], [66, 1, 67, 68, 69, 70, 71], [72, 73]]


In [ ]:
#Padding
max_len=max(len(x) for x in X)
X=[x+[0]*(max_len-len(x)) for x in X]

print(X)

[[2, 3, 0, 0, 0, 0, 0], [4, 5, 0, 0, 0, 0, 0], [6, 7, 3, 0, 0, 0, 0], [1, 9, 10, 0, 0, 0, 0], [11, 5, 0, 0, 0, 0, 0], [12, 3, 0, 0, 0, 0, 0], [13, 14, 0, 0, 0, 0, 0], [15, 16, 0, 0, 0, 0, 0], [17, 18, 0, 0, 0, 0, 0], [19, 20, 0, 0, 0, 0, 0], [21, 22, 23, 24, 0, 0, 0], [25, 26, 5, 1, 28, 29, 30], [31, 32, 33, 0, 0, 0, 0], [21, 34, 35, 36, 0, 0, 0], [37, 38, 1, 39, 40, 0, 0], [41, 3, 0, 0, 0, 0, 0], [42, 5, 0, 0, 0, 0, 0], [6, 43, 3, 0, 0, 0, 0], [1, 44, 10, 0, 0, 0, 0], [45, 5, 0, 0, 0, 0, 0], [46, 3, 0, 0, 0, 0, 0], [47, 48, 0, 0, 0, 0, 0], [49, 50, 51, 0, 0, 0, 0], [52, 53, 0, 0, 0, 0, 0], [54, 55, 10, 0, 0, 0, 0], [21, 56, 57, 35, 58, 0, 0], [59, 60, 61, 62, 0, 0, 0], [25, 63, 64, 65, 0, 0, 0], [66, 1, 67, 68, 69, 70, 71], [72, 73, 0, 0, 0, 0, 0]]


In [ ]:
from sklearn.model_selection import train_test_split
import torch

X=torch.tensor(X)
Y=torch.tensor(labels)

# Split data into training and testing sets, ensuring a stratified split
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y)

train_loader=DataLoader(TensorDataset(X_train,Y_train),batch_size=4,shuffle=True)
test_loader=DataLoader(TensorDataset(X_test,Y_test),batch_size=4,shuffle=False)

In [ ]:
class RNNClassifier(nn.Module):
  def __init__(self):
    super().__init__()

    self.embedding=nn.Embedding(
        len(vocab),16
    )
    self.rnn=nn.RNN(
        16,32,batch_first=True
    )

    self.fc=nn.Linear(32,2)

  def forward(self,x):
    x=self.embedding(x)
    _,h=self.rnn(x)
    return self.fc(h[-1])

In [ ]:
model=RNNClassifier()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

In [ ]:
epochs=50
for epoch in range(epochs):
  model.train()
  correct=0
  total=0
  for x,y in train_loader:
    output=model(x)
    loss=criterion(output,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    prediction=output.argmax(1)

  correct+=(prediction==y).sum().item()
  total+=y.size(0)

  accuracy=100*correct/total
  if(epoch+1)%5==0 and epoch<21:
      print("Epoch:",epoch+1,"Loss:",loss.item(),"Training Accuracy:",round(accuracy,2),"%")

Epoch: 5 Loss: 0.004696173127740622 Training Accuracy: 100.0 %
Epoch: 10 Loss: 0.011633532121777534 Training Accuracy: 100.0 %
Epoch: 15 Loss: 0.010626045055687428 Training Accuracy: 100.0 %
Epoch: 20 Loss: 0.005070090293884277 Training Accuracy: 100.0 %


In [ ]:
model.eval()
correct=0
total=0
all_predictions = []
all_labels = []
with torch.no_grad():
  for x, y in test_loader:
    output = model(x)
    prediction = output.argmax(1)
    correct += (prediction == y).sum().item()
    total += y.size(0)
    all_predictions.extend(prediction.cpu().numpy())
    all_labels.extend(y.cpu().numpy())

  accuracy = 100 * correct / total
  print(f"Testing Accuracy: {accuracy:.2f}%")
  print(f"Test Predictions: {all_predictions}")
  print(f"Actual Test Labels: {all_labels}")

Testing Accuracy: 11.11%
Test Predictions: [np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0)]
Actual Test Labels: [np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1)]
